# Dataset Preparation - NEU Surface Defect Database

Este notebook reorganiza el dataset NEU de 6 categorías originales a nuestras 4 categorías del proyecto:

**Mapeo de categorías:**
- `pitted_surface` → `pitting_corrosion`
- `inclusion` + `patches` → `surface_inclusions`
- `crazing` + `scratches` → `cracks_scratches`
- `rolled-in_scale` → `flawless_prime`

**División del dataset:**
- Train: 70% (1260 imágenes)
- Validation: 15% (270 imágenes)
- Test: 15% (270 imágenes)

In [1]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
import random

# Set random seed for reproducibility
random.seed(42)

In [2]:
# Define paths
BASE_DIR = Path(r'C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos')
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw' / 'NEU' / 'NEU-DET'
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'

print(f"Base directory: {BASE_DIR}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"\nRaw data exists: {RAW_DATA_DIR.exists()}")

Base directory: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos
Raw data directory: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\raw\NEU\NEU-DET
Processed data directory: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed

Raw data exists: True


In [3]:
# Category mapping from original 6 to our 4 categories
CATEGORY_MAPPING = {
    'pitted_surface': 'pitting_corrosion',
    'inclusion': 'surface_inclusions',
    'patches': 'surface_inclusions',
    'crazing': 'cracks_scratches',
    'scratches': 'cracks_scratches',
    'rolled-in_scale': 'flawless_prime'
}

print("Category Mapping:")
for orig, new in CATEGORY_MAPPING.items():
    print(f"  {orig:20} → {new}")

Category Mapping:
  pitted_surface       → pitting_corrosion
  inclusion            → surface_inclusions
  patches              → surface_inclusions
  crazing              → cracks_scratches
  scratches            → cracks_scratches
  rolled-in_scale      → flawless_prime


In [4]:
# Collect all images from train and validation folders
all_images = defaultdict(list)

for split in ['train', 'validation']:
    split_dir = RAW_DATA_DIR / split / 'images'
    
    for orig_category in CATEGORY_MAPPING.keys():
        category_dir = split_dir / orig_category
        
        if category_dir.exists():
            images = list(category_dir.glob('*.jpg')) + list(category_dir.glob('*.bmp'))
            new_category = CATEGORY_MAPPING[orig_category]
            all_images[new_category].extend(images)
            print(f"Found {len(images)} images in {split}/{orig_category} → {new_category}")

print("\n" + "="*60)
print("Total images per new category:")
for category, images in all_images.items():
    print(f"  {category:25} : {len(images)} images")
print(f"\nGrand total: {sum(len(imgs) for imgs in all_images.values())} images")

Found 240 images in train/pitted_surface → pitting_corrosion
Found 240 images in train/inclusion → surface_inclusions
Found 240 images in train/patches → surface_inclusions
Found 240 images in train/crazing → cracks_scratches
Found 240 images in train/scratches → cracks_scratches
Found 240 images in train/rolled-in_scale → flawless_prime
Found 60 images in validation/pitted_surface → pitting_corrosion
Found 60 images in validation/inclusion → surface_inclusions
Found 60 images in validation/patches → surface_inclusions
Found 60 images in validation/crazing → cracks_scratches
Found 60 images in validation/scratches → cracks_scratches
Found 60 images in validation/rolled-in_scale → flawless_prime

Total images per new category:
  pitting_corrosion         : 300 images
  surface_inclusions        : 600 images
  cracks_scratches          : 600 images
  flawless_prime            : 300 images

Grand total: 1800 images


In [5]:
# Shuffle images for each category
for category in all_images:
    random.shuffle(all_images[category])

print("Images shuffled for random distribution")

Images shuffled for random distribution


In [6]:
# Split images into train (70%), val (15%), test (15%)
splits = {'train': 0.70, 'val': 0.15, 'test': 0.15}
split_data = {split: defaultdict(list) for split in splits.keys()}

for category, images in all_images.items():
    n_total = len(images)
    n_train = int(n_total * splits['train'])
    n_val = int(n_total * splits['val'])
    
    split_data['train'][category] = images[:n_train]
    split_data['val'][category] = images[n_train:n_train + n_val]
    split_data['test'][category] = images[n_train + n_val:]
    
    print(f"{category:25} : Train={len(split_data['train'][category])}, Val={len(split_data['val'][category])}, Test={len(split_data['test'][category])}")

print("\n" + "="*60)
for split in splits.keys():
    total = sum(len(imgs) for imgs in split_data[split].values())
    print(f"Total {split:5} images: {total}")

pitting_corrosion         : Train=210, Val=45, Test=45
surface_inclusions        : Train=420, Val=90, Test=90
cracks_scratches          : Train=420, Val=90, Test=90
flawless_prime            : Train=210, Val=45, Test=45

Total train images: 1260
Total val   images: 270
Total test  images: 270


In [7]:
# Create directory structure
print("Creating directory structure...")

for split in splits.keys():
    for category in all_images.keys():
        target_dir = PROCESSED_DATA_DIR / split / category
        target_dir.mkdir(parents=True, exist_ok=True)
        print(f"  Created: {target_dir}")

print("\nDirectory structure created!")

Creating directory structure...
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\train\pitting_corrosion
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\train\surface_inclusions
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\train\cracks_scratches
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\train\flawless_prime
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\val\pitting_corrosion
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\val\surface_inclusions
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\val\cracks_scratches
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\val\flawless_prime
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\test\pitting_corrosion
  Created: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed\test\surface_incl

In [8]:
# Copy images to new structure
print("Copying images to processed directory...\n")

for split in splits.keys():
    print(f"Processing {split} split:")
    
    for category, images in split_data[split].items():
        target_dir = PROCESSED_DATA_DIR / split / category
        
        for i, src_path in enumerate(images, 1):
            # Create new filename with category prefix
            new_filename = f"{category}_{i:04d}{src_path.suffix}"
            target_path = target_dir / new_filename
            
            # Copy file
            shutil.copy2(src_path, target_path)
        
        print(f"  ✓ {category:25} : {len(images)} images copied")
    
    print()

print("="*60)
print("Dataset preparation complete!")

Copying images to processed directory...

Processing train split:
  ✓ pitting_corrosion         : 210 images copied
  ✓ surface_inclusions        : 420 images copied
  ✓ cracks_scratches          : 420 images copied
  ✓ flawless_prime            : 210 images copied

Processing val split:
  ✓ pitting_corrosion         : 45 images copied
  ✓ surface_inclusions        : 90 images copied
  ✓ cracks_scratches          : 90 images copied
  ✓ flawless_prime            : 45 images copied

Processing test split:
  ✓ pitting_corrosion         : 45 images copied
  ✓ surface_inclusions        : 90 images copied
  ✓ cracks_scratches          : 90 images copied
  ✓ flawless_prime            : 45 images copied

Dataset preparation complete!


In [9]:
# Verify the final structure
print("\nFinal dataset structure verification:\n")

for split in splits.keys():
    split_dir = PROCESSED_DATA_DIR / split
    print(f"{split.upper()} split:")
    
    total_split = 0
    for category_dir in sorted(split_dir.iterdir()):
        if category_dir.is_dir():
            count = len(list(category_dir.glob('*')))
            total_split += count
            print(f"  {category_dir.name:25} : {count:4d} images")
    
    print(f"  {'TOTAL':25} : {total_split:4d} images\n")

print("="*60)
print("✅ Dataset is ready for training!")
print(f"\nProcessed data location: {PROCESSED_DATA_DIR}")


Final dataset structure verification:

TRAIN split:
  cracks_scratches          :  420 images
  flawless_prime            :  210 images
  pitting_corrosion         :  210 images
  surface_inclusions        :  420 images
  TOTAL                     : 1260 images

VAL split:
  cracks_scratches          :   90 images
  flawless_prime            :   45 images
  pitting_corrosion         :   45 images
  surface_inclusions        :   90 images
  TOTAL                     :  270 images

TEST split:
  cracks_scratches          :   90 images
  flawless_prime            :   45 images
  pitting_corrosion         :   45 images
  surface_inclusions        :   90 images
  TOTAL                     :  270 images

✅ Dataset is ready for training!

Processed data location: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed


In [11]:
# Display sample image paths
print("\nSample image paths:")
for split in ['train', 'val', 'test']:
    split_dir = PROCESSED_DATA_DIR / split
    sample_image = next(split_dir.rglob('*.jpg'), None) or next(split_dir.rglob('*.bmp'), None)
    if sample_image:
        print(f"  {split:5} : {sample_image.relative_to(PROCESSED_DATA_DIR)}")


Sample image paths:
  train : train\cracks_scratches\cracks_scratches_0001.jpg
  val   : val\cracks_scratches\cracks_scratches_0001.jpg
  test  : test\cracks_scratches\cracks_scratches_0001.jpg
